# PointNet++ v8b — ArcFace Lab (loss ladder + QA-ArcFace) — Track C (Colab terpisah) (R1/R2/R3) — Train + Eval + Analysis

**Tujuan v8b**: Membandingkan **3 representasi input point-cloud** dengan loss
**dikunci** ke `arcface_m04` (margin=0.4, scale=30), 5 seed, **closed-set** (Test+Holdout, **TANPA LOSO**).

**3 representasi yang diuji:**
- **C1 = `raw_ply`** (R1, `output.ply`) — koordinat kamera asli, **tanpa kanonikalisasi**, frame_mode=all
- **C2 = `canonical_npy`** (R2, `cnn_input.npy`) — PCA-align + unit-sphere, frame_mode=all
- **C3 = `fps_npy`** (R3, `cnn_input_fps.npy`) — R2 + FPS fixed 8192, frame_mode=all

**C0 = anchor `arcface_m04` dari v7.1.1** (R2, median frame) — **TIDAK ditraining ulang**,
hanya dimuat sebagai referensi (jika hasilnya tersedia).

**Parameter dikunci**: loss=arcface, margin=0.4, scale=30, `n_points=8192`, seeds=`[0,42,123,2024,31337]`.
R3 fixed 8192; R1/R2 di-subsample ke 8192 agar perbandingan adil.

**Output**: `runs/v8b_arclab/{C1,C2,C3}/seed_*/`, `eval_results/v8b_arclab/`, `analysis/v8b_arclab_<TS>/`.


## 1. Setup — GitHub Clone + Dataset in Repo

In [ ]:
from google.colab import userdata
import os, sys, subprocess
from pathlib import Path

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/RZulfikri/Thesis.git'
REPO_DIR     = Path('/content/Thesis')
PROJECT_ROOT = REPO_DIR / '3DCNN'
COLAB_BRANCH = 'main'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin

os.chdir(str(REPO_DIR))
ret = subprocess.run(['git', 'checkout', COLAB_BRANCH], capture_output=True)
if ret.returncode != 0:
    !git checkout -b {COLAB_BRANCH}
else:
    !git merge origin/main --no-edit --strategy-option theirs 2>/dev/null || true

!git config user.email 'colab-runner@thesis.local'
!git config user.name  'Colab Runner'

# v8b: open3d dibutuhkan untuk repr_mode=raw_ply (R1) — load output.ply.
#         C2/C3 (npy) tidak butuh ini, tapi install tetap aman.
try:
    import open3d  # noqa: F401
except ImportError:
    print('Installing open3d (untuk R1 raw_ply)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'open3d'])
    import open3d  # noqa: F401
print(f'open3d OK ({open3d.__version__})')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))

# Verifikasi dataset & subjek
from utils.dataset_lowdata import scan_sessions, DROPPED_SUBJECTS
sessions = scan_sessions('dataset')
print(f'Subjek aktif: {sorted(sessions.keys())} ({len(sessions)} subjek)')
print(f'DROPPED_SUBJECTS: {DROPPED_SUBJECTS}')
print(f'Total sesi: {sum(len(v) for v in sessions.values())}')
print('Setup OK')

## 2. Konfigurasi — Representasi, Seeds, Loss (dikunci), Direktori

In [ ]:
import torch
import numpy as np
import json
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Opsi B: checkpoint & cache eval ke Google Drive (persist + resume; repo tetap ramping) ---
# Selama run, hasil BESAR (checkpoint .pth + cache .pkl) ditulis ke Drive → tahan disconnect
# & bisa resume, TANPA pernah masuk git. Yang di-commit ke git hanya evidence KECIL (ANA_DIR
# = analysis/, CSV+figur). Arsip final ke Release lewat sel terakhir (opsional).
from google.colab import drive as _gd
_gd.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/PointNetPalm')   # ubah jika ingin folder lain
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_SLUG = 'RZulfikri/Thesis'
print(f'Drive: {DRIVE_ROOT}')

# --- 3 representasi yang dibandingkan (loss DIKUNCI arcface_m04) ---
FIXED_ALIGN = 'align_pca_robust'   # alignment terbaik dari Study A (dikunci utk Track C)
# Tangga loss (head margin) — cfg_id : (loss_type, margin, variant)
LOSS_MAP = {
    'L_softmax':    ('arcface_true',      0.0,  'true'),   # m=0 ~ scaled-softmax (baseline)
    'L_arcface':    ('arcface_true',      0.5,  'true'),   # ArcFace sejati
    'L_cosface':    ('cosface',           0.35, 'true'),
    'L_subcenter':  ('subcenter_arcface', 0.5,  'true'),
    'L_adacos':     ('adacos',            0.0,  'true'),
    'L_curricular': ('curricularface',    0.5,  'true'),
    'L_qa':         ('qa_arcface',        0.5,  'true'),   # USULAN: quality-adaptive
}
REPR_CONFIGS = [(cid, FIXED_ALIGN) for cid in LOSS_MAP]   # eval reuse: load cfg_id, repr=FIXED_ALIGN

SEEDS = [0, 42, 123, 2024, 31337]

# --- Loss dikunci: arcface m=0.4, s=30 ---
LOSS_TYPE   = 'arcface'
ARC_MARGIN  = 0.4
ARC_SCALE   = 30.0

N_POINTS = 8192            # R3 fixed 8192; R1/R2 subsample ke 8192 (perbandingan adil)
FRAME_MODE = 'all'         # frame_mode=all (low-data all-frame regime)

# Multi-frame ablation grid (closed-set)
N_LIST = [1, 3, 5, 10]     # frames untuk enrollment
M_LIST = [1, 3, 5, 10]     # frames untuk probe
N_BEST = 5
M_BEST = 5

EPOCHS          = 120
FINETUNE_EPOCHS = 30

# LR DIKUNCI ke protokol anchor v7.1.1 (lr=1e-3, finetune 1e-4).
# train.py auto-config memakai lr=2e-3 di kelas A100 — tanpa override ini,
# pindah GPU diam-diam mengubah protokol optimisasi.
LR    = 1e-3
FT_LR = 1e-4

# --- Anchor C0 (reuse dari v7.1.1, R2 median) — referensi saja ---
C0_VARIANT = 'arcface_m04'
C0_RUNS_DIR = PROJECT_ROOT / 'runs' / 'v7_1_1' / C0_VARIANT
C0_EVAL_DIR = PROJECT_ROOT / 'eval_results' / 'v7_1_1' / C0_VARIANT

# --- Direktori v8b ---
DATA_DIR = PROJECT_ROOT / 'dataset'
RUNS_DIR = DRIVE_ROOT / 'runs' / 'v8b_arclab'
EVAL_DIR = DRIVE_ROOT / 'eval_results' / 'v8b_arclab'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'REPR_CONFIGS: {REPR_CONFIGS}')
print(f'Loss (dikunci): {LOSS_TYPE} m={ARC_MARGIN} s={ARC_SCALE}')
print(f'Seeds: {SEEDS}  |  N_POINTS: {N_POINTS}  |  frame_mode: {FRAME_MODE}')

## 2c. Generate Dataset dari Raw Depth Data (reproducible) — sekali per sesi


In [ ]:
# §2c — Generate dataset dari "Raw Depth Data" (REPRODUCIBLE; repo tak menyimpan dataset).
# Raw scan iPhone (~91MB ZIP) ada di repo → bangun ulang output.ply/geometry/cnn_input/
# fps/align via generate_dataset.py. Idempotent: skip bila dataset sudah ada di sesi ini.
import subprocess as _sp, sys as _sys, glob as _glob
_have = len(_glob.glob(str(DATA_DIR / '*/*/frame_*/output.ply')))
if _have > 0:
    print(f'[data] dataset sudah ada ({_have} frame) - skip generate.')
else:
    print('[data] generate dataset dari Raw Depth Data (~35-90 mnt, sekali per sesi) ...')
    # deps pipeline (torch/numpy/pandas sudah ada di Colab); pasang yg kurang saja.
    _sp.run([_sys.executable, '-m', 'pip', 'install', '-q',
             'open3d', 'opencv-python-headless', 'scikit-learn', 'scipy'], check=True)
    _r = _sp.run([_sys.executable, 'generate_dataset.py'], cwd=str(REPO_DIR / '3DRegistration'))
    assert _r.returncode == 0, 'generate_dataset.py gagal - cek log di atas.'
_have = len(_glob.glob(str(DATA_DIR / '*/*/frame_*/output.ply')))
print(f'[data] dataset siap: {_have} frame output.ply di {DATA_DIR}')
assert _have > 0, 'dataset kosong setelah generate - cek Raw Depth Data & generate_dataset.py'


### 2b. Timestamp Analisis (jalankan sekali)

In [ ]:
# TS didefinisikan saat runtime (bukan saat import). Jalankan cell ini sekali.
from datetime import datetime
TS = datetime.now().strftime('%Y%m%d_%H%M%S')
ANA_DIR = PROJECT_ROOT / 'analysis' / f'v8b_arclab_{TS}'
ANA_DIR.mkdir(parents=True, exist_ok=True)
print(f'TS = {TS}')
print(f'ANA_DIR = {ANA_DIR}')

## 3. GPU Probe — Batch Size Auto-Tune + AMP Detection

In [ ]:
import subprocess, os, sys, gc

TARGET_VRAM_FRACTION = 0.75
MAX_BS_FOR_LARGE_N   = 192
# bf16 hanya untuk Ampere+ (CC>=8: A100/L4/H100). is_bf16_supported() di torch baru
# mengembalikan True di T4 (emulasi, tanpa tensor core — lambat); T4/V100 harus fp16.
_CC = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
AMP_MODE             = 'bf16' if _CC >= 8 else 'fp16'
PRELOAD_AUGMENT      = True
NUM_WORKERS          = 8

try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader,nounits'], text=True
    )
    line = out.strip().split('\n')[0]
    vram_str, gpu_name = [x.strip() for x in line.split(',')]
    VRAM_GB = int(vram_str) / 1024
    GPU_NAME = gpu_name
except Exception:
    VRAM_GB = 0; GPU_NAME = 'Unknown'

target_vram_gb = VRAM_GB * TARGET_VRAM_FRACTION
print(f'GPU: {GPU_NAME} ({VRAM_GB:.1f} GB) | target={target_vram_gb:.1f} GB | AMP={AMP_MODE}')

def probe_max_batch_size(n_points, target_gb, amp_dtype):
    sys.path.insert(0, str(PROJECT_ROOT))
    from models.siamese import SiamesePalmNet
    device = torch.device('cuda')
    def _try_bs(bs):
        torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
        try:
            model = SiamesePalmNet(geom_dim=13, use_geom=False, use_aux_loss=False,
                                   n_subjects=11, siamese_mode='concat').to(device)
            model.train()
            opt = torch.optim.Adam(model.parameters(), lr=1e-3, fused=True)
            pts  = torch.randn(bs, n_points, 6, device=device)
            geom = torch.randn(bs, 13, device=device)
            ctx  = torch.amp.autocast('cuda', dtype=amp_dtype) if amp_dtype else torch.amp.autocast('cuda', enabled=False)
            with ctx:
                emb = model.encode(pts, geom); loss = emb.sum()
            loss.backward(); opt.step(); torch.cuda.synchronize()
            peak = torch.cuda.max_memory_allocated() / (1024**3)
            del model, opt, pts, geom, emb, loss; torch.cuda.empty_cache(); gc.collect()
            return peak
        except (torch.cuda.OutOfMemoryError, RuntimeError):
            torch.cuda.empty_cache(); gc.collect(); return float('inf')
    amp_dt = torch.bfloat16 if AMP_MODE == 'bf16' else (torch.float16 if AMP_MODE == 'fp16' else None)
    bs_low, best_bs, best_peak = 32, 32, 0.0
    bs = 32
    while True:
        peak = _try_bs(bs)
        if peak == float('inf'):
            bs_high = bs; break
        best_bs, best_peak = bs, peak
        if peak >= target_gb * 0.95:
            bs_high = int(bs * (target_gb / peak) * 1.1); break
        bs_low = bs; bs *= 2
    else:
        bs_high = bs_low * 2
    while bs_high - bs_low > 16:
        m = (bs_low + bs_high) // 2
        peak = _try_bs(m)
        if peak == float('inf') or peak > VRAM_GB * 0.95:
            bs_high = m
        else:
            if peak <= target_gb:
                best_bs, best_peak = m, peak; bs_low = m
            else:
                bs_high = m
    final_bs = max(32, int(best_bs * 0.9))
    final_bs = min(final_bs, MAX_BS_FOR_LARGE_N)
    print(f'  -> BS={final_bs}, peak~{best_peak:.1f}GB')
    return final_bs, best_peak

amp_dt = torch.bfloat16 if AMP_MODE == 'bf16' else (torch.float16 if AMP_MODE == 'fp16' else None)
if torch.cuda.is_available():
    BS, peak_gb = probe_max_batch_size(N_POINTS, target_vram_gb, amp_dt)
else:
    BS, peak_gb = 32, 0.0

# 11 subjek × 8 sesi training (low-data); repeat untuk panjang dataset
REPEAT = max(4, -(-min(BS, 512) * 4 // 88))

print('\n' + '='*60)
print(f'AUTO-TUNE: BS={BS}  REPEAT={REPEAT}  N_POINTS={N_POINTS}  AMP={AMP_MODE}')
print('='*60)
torch.cuda.empty_cache(); gc.collect()

## 4. Helper: `run_training()` (loss dikunci arcface_m04 + --repr-mode / --frame-mode)

In [ ]:
def run_training(cfg_id, repr_mode, seed):
    loss_type, _m, _variant = LOSS_MAP[cfg_id]; repr_mode = FIXED_ALIGN
    out_dir = RUNS_DIR / cfg_id / f'seed_{seed}'
    ckpt    = out_dir / 'best.pth'
    if ckpt.exists():
        print(f'  SKIP training {cfg_id} ({repr_mode}) seed={seed} (done)')
        return True

    amp_flag     = f'--amp {AMP_MODE}' if AMP_MODE != 'none' else ''
    preload_flag = '--preload-augment' if PRELOAD_AUGMENT else ''

    cmd = (
        f'python {PROJECT_ROOT / "train.py"} '
        f'--data_dir {DATA_DIR} '
        f'--output_dir {out_dir} '
        f'--seed {seed} '
        f'--fixed_split '
        f'--frames-per-session 1 '
        f'--frame-mode {FRAME_MODE} '
        f'--repr-mode {repr_mode} '
        f'--loss {loss_type} --arcface-margin {_m} --arcface-scale {ARC_SCALE} --arcface-variant {_variant} '
        f'--val-metric pair_eer --no-early-stop '
        f'--epochs {EPOCHS} --finetune_epochs {FINETUNE_EPOCHS} '
        f'--lr {LR} --finetune_lr {FT_LR} '
        f'--batch_size {BS} --n_points {N_POINTS} '
        f'--num_workers {NUM_WORKERS} '
        f'--repeat {REPEAT} '
        f'{amp_flag} {preload_flag} '
        f'--siamese-mode concat'
    )
    print(f'\n{"="*60}')
    print(f'TRAIN: {cfg_id} | repr={repr_mode} | seed={seed} | loss={LOSS_TYPE} m={ARC_MARGIN} s={ARC_SCALE}')
    print(f'{"="*60}')
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / 'train_stdout.log'
    # tee: tampilkan live DAN simpan log (agar error/traceback ter-commit & bisa diperiksa)
    !{cmd} 2>&1 | tee "{log_path}"
    ok = ckpt.exists()
    if not ok:
        print(f'  [GAGAL] {cfg_id} seed={seed}: best.pth tidak terbentuk — cek {log_path}')
    # Post-check spec: SafetyClamp train.py bisa diam-diam menurunkan n_points
    # (run T4 2026-06-12: 8192 -> 4096). Hasil di luar spec = tidak dipakai.
    perf_path = out_dir / 'perf.json'
    if ok and perf_path.exists():
        perf = json.loads(perf_path.read_text())
        if perf.get('n_points') != N_POINTS:
            ok = False
            print(f'  [SPEC-MISMATCH] {cfg_id} seed={seed}: n_points aktual '
                  f'{perf.get("n_points")} != spec {N_POINTS} (GPU di-clamp?). '
                  f'Hasil ditandai GAGAL — pakai GPU >= A100 40GB.')
    # commit+push per seed: log/checkpoint langsung terlihat di GitHub
    # tanpa menunggu 5 seed config selesai (cell training berjalan berjam-jam)
    git_save(f'v8b [auto] {cfg_id} ({repr_mode}) seed={seed}: {"OK" if ok else "GAGAL"}',
             push=True)
    return ok

print('run_training ready (v8b)')

## 5. Runtime Shutdown Guard + Git Save

In [ ]:
import atexit, subprocess
_shutdown_triggered = False

def shutdown_colab(silent=False):
    global _shutdown_triggered
    if _shutdown_triggered: return
    _shutdown_triggered = True
    if not silent:
        print('Shutting down Colab runtime...')
    try:
        from google.colab import runtime; runtime.unassign()
    except Exception as e:
        if not silent: print(f'Shutdown error: {e}')

atexit.register(shutdown_colab, silent=True)

def _git(args, timeout=600):
    """git dgn TIMEOUT -> tidak pernah nge-hang notebook. Return (rc, out)."""
    try:
        r = subprocess.run(['git'] + args, cwd=str(REPO_DIR),
                           capture_output=True, text=True, timeout=timeout)
        return r.returncode, ((r.stdout or '') + (r.stderr or ''))
    except subprocess.TimeoutExpired:
        return 124, f'TIMEOUT {timeout}s: git {" ".join(args)}'

def git_save(message, push=False, retries=2, timeout=600):
    """Commit & push semua file. Tahan HANG (timeout) & non-fast-forward (sync dulu).
    Tidak pernah raise / nge-hang: kegagalan dicetak, commit aman lokal, dicoba lagi
    di git_save berikutnya."""
    os.chdir(str(REPO_DIR))
    _git(['add', '-A'])
    if subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=str(REPO_DIR)).returncode == 0:
        print('  (nothing to commit)'); os.chdir(str(PROJECT_ROOT)); return
    rc, out = _git(['commit', '-m', message])
    print(f'Committed: {message}' if rc == 0 else f'Commit error: {out[:300]}')
    if push:
        for attempt in range(1, retries + 2):
            # sinkronkan dulu: kalau colab sudah maju, gabung (menangkan versi run ini
            # bila ada konflik) supaya push tidak ditolak non-fast-forward
            _git(['fetch', 'origin', COLAB_BRANCH], timeout=timeout)
            _git(['merge', '-X', 'ours', '--no-edit', f'origin/{COLAB_BRANCH}'], timeout=timeout)
            rc, out = _git(['push', 'origin', COLAB_BRANCH], timeout=timeout)
            if rc == 0:
                print('Pushed OK'); break
            print(f'Push gagal (attempt {attempt}/{retries + 1}): {out[:200]}')
        else:
            print('  [WARN] push tetap gagal - commit AMAN lokal, akan dicoba lagi di cell berikutnya.')
    os.chdir(str(PROJECT_ROOT))

print('Shutdown guard + git_save (tahan hang & non-ff) ready')

## 5c. Auto-Commit per Cell — Snapshot Notebook + Push

Setiap cell selesai dieksekusi, hook `post_run_cell`:
1. mengambil JSON notebook **termasuk output cell** dari frontend Colab
   (`google.colab._message.blocking_request('get_ipynb')`),
2. menulisnya ke `collab/snapshots/v8b_arclab_repr_ablation_LIVE.ipynb`
   (output panjang dipangkas ke 200 baris terakhir — log penuh tetap di
   `train_stdout.log`),
3. `git_save(push=True)` → progres bisa dipantau dari GitHub tanpa membuka Colab.

Min interval antar auto-commit 60 dtk (cell-cell cepat digabung ke commit berikutnya).

In [ ]:
import time
import json as _json
from IPython import get_ipython

SNAPSHOT_PATH = PROJECT_ROOT / 'collab' / 'snapshots' / 'v8b_arclab_repr_ablation_LIVE.ipynb'
SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
AUTOCOMMIT_MIN_INTERVAL = 60  # detik
_last_autocommit = [0.0]


def _trim_outputs(nb_json, max_lines=200):
    """Pangkas output stream yang panjang agar snapshot tidak membengkak."""
    for cell in nb_json.get('cells', []):
        for out in cell.get('outputs') or []:
            txt = out.get('text')
            if isinstance(txt, list) and len(txt) > max_lines:
                out['text'] = (['... (dipangkas — log penuh di train_stdout.log) ...\n']
                               + txt[-max_lines:])
    return nb_json


def snapshot_notebook():
    """Tulis state notebook saat ini (dengan output) ke SNAPSHOT_PATH."""
    try:
        from google.colab import _message
        resp = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
        nb_json = (resp or {}).get('ipynb')
        if not nb_json:
            return False
        SNAPSHOT_PATH.write_text(
            _json.dumps(_trim_outputs(nb_json), ensure_ascii=False, indent=1))
        return True
    except Exception as e:  # di luar Colab / frontend tidak merespons → no-op
        print(f'  [autocommit] snapshot gagal: {e}')
        return False


def _post_run_cell(result):
    # Hook TIDAK BOLEH mematahkan eksekusi — semua error ditelan dengan pesan.
    try:
        now = time.time()
        if now - _last_autocommit[0] < AUTOCOMMIT_MIN_INTERVAL:
            return
        _last_autocommit[0] = now
        raw = getattr(getattr(result, 'info', None), 'raw_cell', '') or ''
        head = next((l.strip() for l in raw.splitlines() if l.strip()), '?')[:60]
        snapshot_notebook()
        git_save(f'v8b [auto] cell selesai: {head}', push=True)
    except Exception as e:
        print(f'  [autocommit] error: {e}')


_ip = get_ipython()
# hindari registrasi ganda bila cell ini dijalankan ulang
for _cb in list(_ip.events.callbacks.get('post_run_cell', [])):
    if getattr(_cb, '__name__', '') == '_post_run_cell':
        _ip.events.unregister('post_run_cell', _cb)
_ip.events.register('post_run_cell', _post_run_cell)
print(f'Auto-commit per cell AKTIF -> {SNAPSHOT_PATH.relative_to(PROJECT_ROOT)} '
      f'(min interval {AUTOCOMMIT_MIN_INTERVAL}s)')

## 5a. Verifikasi Varian Alignment (align_*.npy) — dari Release; fallback generate lokal


In [ ]:
# §5a — Pastikan varian alignment (align_*.npy) lengkap.
# Dataset (termasuk align) seharusnya SUDAH ada dari §2c (Release). Fallback bila Release
# dibuat SEBELUM align digenerate: generate lokal (idempotent, TANPA commit — dataset
# di-gitignore; sumber kebenaran ada di Release, bukan git).
import glob as _glob
_n_ply = len(_glob.glob(str(DATA_DIR / '*/*/frame_*/output.ply')))
_modes = ['align_center', 'align_centerscale', 'align_pca_robust', 'align_anatomical']

def _align_complete():
    return all(
        len(_glob.glob(str(DATA_DIR / f'*/*/frame_*/{m}.npy'))) == _n_ply
        for m in _modes
    )

if _align_complete():
    print(f'[5a] align_*.npy lengkap dari Release ({_n_ply} frame x {len(_modes)} mode).')
else:
    print('[5a] align_*.npy belum lengkap di Release - generate lokal (TANPA commit) ...')
    !cd {REPO_DIR / '3DRegistration'} && python make_align_variants.py --data_dir {DATA_DIR}
    assert _align_complete(), 'align_*.npy belum lengkap - cek make_align_variants.py'
    print('[5a] selesai generate lokal (pertimbangkan re-pack Release agar tak ulang).')

for m in _modes:
    print(f'  {m}.npy: {len(_glob.glob(str(DATA_DIR / f"*/*/frame_*/{m}.npy")))}/{_n_ply}')


## 5b. Preflight — Kelengkapan File Representasi (WAJIB sebelum training)

Run 2026-06-08 gagal di 5 seed C1 karena `output.ply` tidak ada di checkout
(`3DCNN/dataset/**/*.ply` ter-gitignore; hanya 424/2.131 PLY yang ter-push).

Cell ini memverifikasi file representasi tiap config dan menghasilkan
**`RUNNABLE_CONFIGS`** — config yang datanya tidak lengkap **di-SKIP dengan
peringatan** (bukan memblokir semuanya), sehingga C2/C3 tetap bisa jalan
selagi PLY C1 belum di-push. Training abort hanya bila TIDAK ADA config
yang lengkap. Config yang ter-skip otomatis ikut di run berikutnya setelah
datanya lengkap (seed yang sudah punya `best.pth` tidak di-retrain).

In [ ]:
# Preflight: filter config yang file representasinya lengkap di SEMUA frame dir.
# Hasil: RUNNABLE_CONFIGS (dipakai loop training). Config tidak lengkap di-skip.
from utils.dataset import _REPR_FILE

# Guard GPU: spec v8b (n_points=8192, batch 192) butuh kelas A100 40GB+.
# Di GPU lebih kecil SafetyClamp train.py menurunkan n_points DIAM-DIAM ke 4096
# (terjadi di run T4 2026-06-12) — hasilnya menyimpang dari spec & dibuang.
ALLOW_SPEC_CLAMP = False   # set True HANYA bila sengaja meresmikan spec lebih rendah
if not ALLOW_SPEC_CLAMP and N_POINTS > 4096 and VRAM_GB < 35:
    raise RuntimeError(
        f'GPU {GPU_NAME} ({VRAM_GB:.0f} GB) akan meng-clamp n_points {N_POINTS} -> 4096. '
        f'Ganti runtime ke A100 (Runtime > Change runtime type), atau set '
        f'ALLOW_SPEC_CLAMP=True bila deviasi memang disengaja.'
    )

frame_dirs = sorted(DATA_DIR.glob('*/*/frame_*'))
assert frame_dirs, f'Tidak ada frame dir di {DATA_DIR}'
print(f'Total frame dir: {len(frame_dirs)}')

RUNNABLE_CONFIGS, _skipped = [], []
for cfg_id, repr_mode in REPR_CONFIGS:
    fname = _REPR_FILE[repr_mode]
    missing = [d for d in frame_dirs if not (d / fname).exists()]
    status = 'OK' if not missing else f'KURANG {len(missing)}/{len(frame_dirs)} -> SKIP'
    print(f'  {cfg_id} ({repr_mode:14s} -> {fname:18s}): {status}')
    if missing:
        per_sub = {}
        for d in missing:
            per_sub[d.parts[-3]] = per_sub.get(d.parts[-3], 0) + 1
        print(f'      missing per subjek: {per_sub}')
        _skipped.append(cfg_id)
    else:
        RUNNABLE_CONFIGS.append((cfg_id, repr_mode))

if not RUNNABLE_CONFIGS:
    raise RuntimeError(
        'TIDAK ADA config dengan dataset lengkap - training dibatalkan. '
        "PLY ter-gitignore: push dari mesin lokal dengan git add -f "
        "'3DCNN/dataset/**/*.ply' lalu jalankan ulang notebook."
    )
if _skipped:
    print(f'\n!!! Config DI-SKIP run ini: {_skipped} - dataset tidak lengkap.')
    print("!!! Untuk C1: push PLY dari mesin lokal (git add -f '3DCNN/dataset/**/*.ply'),")
    print('!!! lalu jalankan ulang notebook - seed yang sudah selesai tidak di-retrain.')
print(f'\nPreflight: {len(RUNNABLE_CONFIGS)}/{len(REPR_CONFIGS)} config siap -> '
      f'{[c for c, _ in RUNNABLE_CONFIGS]}')

## 6. Training — Representasi (RUNNABLE_CONFIGS) × 5 Seeds

Maksimal 3 config (C1/C2/C3) × 5 seeds = 15 run; yang dijalankan hanya
`RUNNABLE_CONFIGS` hasil preflight §5b. Loss dikunci `arcface_m04`.
`git_save` per seed (di `run_training`) + per config (loop ini).

In [ ]:
# RUNNABLE_CONFIGS berasal dari preflight 5b (NameError di sini = preflight belum dijalankan)
for cfg_id, repr_mode in RUNNABLE_CONFIGS:
    print(f'\n### Config: {cfg_id} (repr={repr_mode}) ###')
    n_ok = sum(bool(run_training(cfg_id, repr_mode, seed)) for seed in SEEDS)
    status = 'complete' if n_ok == len(SEEDS) else f'FAILED ({n_ok}/{len(SEEDS)} seeds OK)'
    git_save(f'v8b: training {cfg_id} ({repr_mode}) {status} ({len(SEEDS)} seeds)', push=True)

## 7. Helper: Load Model (v8b + anchor C0 dari v7.1.1)

In [ ]:
from models.siamese import SiamesePalmNet
from utils.normalizer import GeometryNormalizer

def _build_model():
    return SiamesePalmNet(
        geom_dim=13, use_geom=False, use_aux_loss=False,
        n_subjects=11, siamese_mode='concat',
    ).to(DEVICE)

def _load_ckpt(model, ckpt_path):
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    state = ckpt.get('model_state_dict', ckpt)
    if any(k.startswith('encoder.proj.') for k in state.keys()):
        state = {k.replace('encoder.proj.', 'encoder.proj_no_geom.'): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.eval()
    return model

def load_model(cfg_id, seed):
    """Load model v8b dari runs/v8b_arclab/{cfg_id}/seed_{seed}/."""
    base = RUNS_DIR / cfg_id / f'seed_{seed}'
    ckpt_path, norm_path = base / 'best.pth', base / 'normalizer.json'
    if not ckpt_path.exists():
        return None, None
    model = _load_ckpt(_build_model(), ckpt_path)
    normalizer = GeometryNormalizer.load(str(norm_path)) if norm_path.exists() else None
    return model, normalizer

def load_model_c0(seed):
    """Load anchor C0 (arcface_m04) dari v7.1.1 — referensi saja."""
    base = C0_RUNS_DIR / f'seed_{seed}'
    ckpt_path, norm_path = base / 'best.pth', base / 'normalizer.json'
    if not ckpt_path.exists():
        return None, None
    model = _load_ckpt(_build_model(), ckpt_path)
    normalizer = GeometryNormalizer.load(str(norm_path)) if norm_path.exists() else None
    return model, normalizer

# Test load
m0, n0 = load_model(REPR_CONFIGS[0][0], SEEDS[0])
print(f'Model {REPR_CONFIGS[0][0]} seed={SEEDS[0]}: {"OK" if m0 is not None else "MISSING"}')
c0m, _ = load_model_c0(SEEDS[0])
print(f'Anchor C0 (v7.1.1 {C0_VARIANT}) seed={SEEDS[0]}: {"OK" if c0m is not None else "MISSING (referensi dilewati)"}')

## 8. Session Splits (closed-set: Test + Holdout)

In [ ]:
from utils.dataset_lowdata import build_lowdata_splits_session_dirs

all_session_splits = build_lowdata_splits_session_dirs(str(DATA_DIR))

for split, subs in all_session_splits.items():
    total = sum(len(v) for v in subs.values())
    print(f'  {split:8s}: {len(subs)} subjek, {total} sesi')

label_names = sorted(all_session_splits['test'].keys())
print(f'\nSubjek: {label_names}')

## 9. ACCURACY — N×M Multi-Frame Ablation per Config

`eval_multiframe_ablation(..., repr_mode=repr_mode)` untuk setiap config/seed atas grid N×M.
Agregasi mean±std lintas seed → DataFrame per config. Simpan `ablation_nm_{C1,C2,C3}.csv`.

In [ ]:
from utils.eval_multiframe import eval_multiframe_ablation
import pickle, os
CACHE_DIR = RUNS_DIR / '_eval_cache'; CACHE_DIR.mkdir(parents=True, exist_ok=True)
_CK = CACHE_DIR / 'ablation_results.pkl'

# RESUME: muat cache (boleh PARTIAL) -> lanjut yg belum; corrupt -> mulai bersih
ablation_results = {}
if _CK.exists():
    try:
        _o = pickle.load(open(_CK, 'rb'))
        if isinstance(_o, dict): ablation_results = _o
        print(f'[resume] ablation_results: {len(ablation_results)} (cfg,seed) dimuat')
    except Exception as _e:
        print(f'[corrupt] ablation_results: {_e} -> mulai bersih'); ablation_results = {}

def _save_ablation():
    _tmp = str(_CK) + '.tmp'
    with open(_tmp, 'wb') as _f: pickle.dump(ablation_results, _f)
    os.replace(_tmp, _CK)   # tulis ATOMIK -> cache tak pernah setengah-jadi

_new = 0
for cfg_id, repr_mode in REPR_CONFIGS:
    print(f'\n=== Ablation N×M: {cfg_id} (repr={repr_mode}) ===')
    for seed in SEEDS:
        if (cfg_id, seed) in ablation_results:
            print(f'  seed={seed}: sudah ada (resume), skip'); continue
        model, normalizer = load_model(cfg_id, seed)
        if model is None:
            print(f'  seed={seed}: checkpoint missing, skip'); continue
        res = eval_multiframe_ablation(
            model=model, enroll_session_splits=all_session_splits['train'],
            probe_session_splits=all_session_splits['test'],
            n_list=N_LIST, m_list=M_LIST, fusion_strategy='mean',
            device=DEVICE, n_points=N_POINTS, normalizer=normalizer,
            seed=seed, repr_mode=repr_mode,
        )
        ablation_results[(cfg_id, seed)] = res
        eer_55 = res.get((5, 5), {}).get('eer', float('nan'))
        eer_11 = res.get((1, 1), {}).get('eer', float('nan'))
        print(f'  seed={seed}: EER(1,1)={eer_11:.4f}  EER(5,5)={eer_55:.4f}')
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        _save_ablation()                                                   # checkpoint disk per (cfg,seed)
        git_save(f'v8b [auto] eval §9 ablation {cfg_id} seed={seed}', push=True)  # commit bertahap
        _new += 1
print(f'\nAblation N×M selesai ({len(ablation_results)} entri, {_new} baru).')


In [ ]:
# Agregat mean±std lintas seed per (N,M) per config + simpan CSV
agg_ablation = {}
for cfg_id, repr_mode in REPR_CONFIGS:
    rows = []
    for n in N_LIST:
        for m in M_LIST:
            eers = []
            for seed in SEEDS:
                res = ablation_results.get((cfg_id, seed), {})
                eer = res.get((n, m), {}).get('eer', float('nan'))
                if not np.isnan(eer):
                    eers.append(eer)
            rows.append({'n_enroll': n, 'm_probe': m,
                         'eer_mean': np.mean(eers) if eers else float('nan'),
                         'eer_std':  np.std(eers)  if eers else float('nan'),
                         'n_seeds':  len(eers)})
    df = pd.DataFrame(rows)
    agg_ablation[cfg_id] = df
    df.to_csv(ANA_DIR / f'ablation_nm_{cfg_id}.csv', index=False)
    pivot = df.pivot(index='n_enroll', columns='m_probe', values='eer_mean')
    print(f'\n=== EER mean (5 seeds) — {cfg_id} ({repr_mode}) ===')
    print(pivot.round(4).to_string())

print(f'\nDisimpan: ablation_nm_{{C1,C2,C3}}.csv di {ANA_DIR}')

## 10. DET / ROC + AUC per Config (N=5, M=5)

Hitung skor genuine/impostor dari `gallery_embs`/`probe_embs` (N5M5, seed pertama yg tersedia),
plot DET (FAR vs FRR, log-scale) dan ROC. Simpan `det_roc.png`.

In [ ]:
def genuine_impostor_scores(gallery_embs, gallery_labels, probe_embs, probe_labels):
    """Cosine similarity probe vs gallery → skor genuine & impostor."""
    g = np.asarray(gallery_embs); p = np.asarray(probe_embs)
    g = g / (np.linalg.norm(g, axis=1, keepdims=True) + 1e-9)
    p = p / (np.linalg.norm(p, axis=1, keepdims=True) + 1e-9)
    sim = p @ g.T
    gen, imp = [], []
    for i, pl in enumerate(probe_labels):
        for j, gl in enumerate(gallery_labels):
            (gen if pl == gl else imp).append(sim[i, j])
    return np.array(gen), np.array(imp)

def det_roc_curves(gen, imp, n_thr=300):
    lo = min(gen.min(), imp.min()); hi = max(gen.max(), imp.max())
    thr = np.linspace(lo, hi, n_thr)
    far = np.array([(imp >= t).mean() for t in thr])   # impostor diterima
    frr = np.array([(gen <  t).mean() for t in thr])   # genuine ditolak
    tar = 1.0 - frr                                     # TAR untuk ROC
    auc = float(np.trapz(tar[::-1], far[::-1]))
    i = int(np.argmin(np.abs(far - frr)))
    eer = float((far[i] + frr[i]) / 2.0)
    return far, frr, tar, auc, eer

def first_available(cfg_id):
    for seed in SEEDS:
        res = ablation_results.get((cfg_id, seed), {}).get((N_BEST, M_BEST))
        if res and 'gallery_embs' in res and 'probe_embs' in res:
            return seed, res
    return None, None

# FIX v8b: eval_multiframe_ablation mengembalikan gallery_embs sebagai
# dict {label: (D,)} dan probe_embs sebagai list [(label, (D,))] — label MENEMPEL
# di datanya. Bongkar jadi matriks (n,D) + list label terpisah dulu.
def _unpack_gp(gallery_dict, probe_list):
    g_labels = list(gallery_dict.keys())
    G = np.stack([np.asarray(gallery_dict[l], dtype=float).ravel() for l in g_labels])
    p_labels = [lab for lab, _ in probe_list]
    P = np.stack([np.asarray(e, dtype=float).ravel() for _, e in probe_list])
    return G, g_labels, P, p_labels

fig, (ax_det, ax_roc) = plt.subplots(1, 2, figsize=(13, 5.5))
auc_summary = {}
for cfg_id, repr_mode in REPR_CONFIGS:
    seed, res = first_available(cfg_id)
    if res is None:
        print(f'  {cfg_id}: tidak ada gallery/probe embs, skip')
        continue
    G, g_labs, P, p_labs = _unpack_gp(res['gallery_embs'], res['probe_embs'])
    gen, imp = genuine_impostor_scores(G, g_labs, P, p_labs)
    if len(gen) == 0 or len(imp) == 0:
        print(f'  {cfg_id}: skor kosong, skip'); continue
    far, frr, tar, auc, eer = det_roc_curves(gen, imp)
    auc_summary[cfg_id] = {'auc': auc, 'eer': eer, 'seed': seed}
    lbl = f'{cfg_id} ({repr_mode}) AUC={auc:.3f} EER={eer:.3f}'
    eps = 1e-3
    ax_det.plot(np.clip(far, eps, 1), np.clip(frr, eps, 1), label=lbl)
    ax_roc.plot(far, tar, label=lbl)

ax_det.set_xscale('log'); ax_det.set_yscale('log')
ax_det.set_xlabel('FAR'); ax_det.set_ylabel('FRR'); ax_det.set_title('DET (N=5, M=5)')
ax_det.grid(True, which='both', alpha=0.3); ax_det.legend(fontsize=8)
ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax_roc.set_xlabel('FAR'); ax_roc.set_ylabel('TAR'); ax_roc.set_title('ROC (N=5, M=5)')
ax_roc.grid(True, alpha=0.3); ax_roc.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ANA_DIR / 'det_roc.png', bbox_inches='tight')
plt.show()
print(f'AUC summary: {auc_summary}')
print(f'Disimpan: {ANA_DIR}/det_roc.png')

## 11. SPEED & RESOURCE Table

Per config: training wall (`perf.json`), peak GPU mem, eval latency (N5M5), disk footprint
per representasi (sum ukuran file `output.ply`/`cnn_input.npy`/`cnn_input_fps.npy` di dataset),
dan microbench preprocess-latency (`load_session` rata-rata ~50 frame). Simpan `speed_resource.csv`.

In [ ]:
import glob, os, time as _time
from utils.dataset import load_session

# File representasi untuk disk footprint
REPR_FILE = {'raw_ply': 'output.ply', 'canonical_npy': 'cnn_input.npy', 'fps_npy': 'cnn_input_fps.npy'}

def disk_footprint_mb(fname):
    total = 0
    for p in glob.glob(str(DATA_DIR / '**' / fname), recursive=True):
        try: total += os.path.getsize(p)
        except OSError: pass
    return total / (1024 ** 2)

def read_perf(cfg_id):
    walls, mems = [], []
    for seed in SEEDS:
        f = RUNS_DIR / cfg_id / f'seed_{seed}' / 'perf.json'
        if not f.exists():
            continue
        try:
            with open(f) as fp: p = json.load(fp)
            if p.get('total_train_wall_s') is not None: walls.append(p['total_train_wall_s'])
            if p.get('peak_gpu_mem_mb')  is not None: mems.append(p['peak_gpu_mem_mb'])
        except Exception as e:
            print(f'  [WARN] gagal baca {f}: {e}')
    return walls, mems

def microbench_load(repr_mode, n_frames=50):
    # kumpulkan frame_dir dari train+test
    frame_dirs = []
    for sp in ('train', 'test'):
        for lab, sessions_ in all_session_splits[sp].items():
            for sdir in sessions_:
                for fd in sorted(Path(sdir).glob('frame_*')):
                    if (fd / 'geometry.json').exists():
                        frame_dirs.append(fd)
    frame_dirs = frame_dirs[:n_frames]
    if not frame_dirs:
        return float('nan')
    t0 = _time.perf_counter()
    ok = 0
    for fd in frame_dirs:
        try:
            load_session(fd, repr_mode=repr_mode); ok += 1
        except Exception:
            pass
    if ok == 0: return float('nan')
    return (_time.perf_counter() - t0) / ok

import pickle, os
CACHE_DIR = RUNS_DIR / '_eval_cache'; CACHE_DIR.mkdir(parents=True, exist_ok=True)
_CK = CACHE_DIR / 'speed_rows.pkl'
speed_rows = None
if _CK.exists():
    try:
        _o = pickle.load(open(_CK, 'rb'))
        if isinstance(_o, list) and len(_o) == len(REPR_CONFIGS):
            speed_rows = _o; print('[checkpoint HIT] speed_rows - selesai, skip')
        else:
            print('[checkpoint INVALID] speed_rows - tak lengkap, hitung ulang')
    except Exception as _e:
        print(f'[checkpoint CORRUPT] speed_rows: {_e} - hitung ulang')
else:
    print('[checkpoint MISS] speed_rows - belum ada, mulai dari awal')
if speed_rows is None:
    speed_rows = []
    for cfg_id, repr_mode in REPR_CONFIGS:
        walls, mems = read_perf(cfg_id)
        # latency N5M5 (mean lintas seed yg ada)
        lat_en, lat_pr = [], []
        for seed in SEEDS:
            r = ablation_results.get((cfg_id, seed), {}).get((N_BEST, M_BEST), {})
            if r.get('latency_enroll_s') is not None: lat_en.append(r['latency_enroll_s'])
            if r.get('latency_probe_s')  is not None: lat_pr.append(r['latency_probe_s'])
        disk_mb = disk_footprint_mb(REPR_FILE[repr_mode])
        prep_s = microbench_load(repr_mode)
        speed_rows.append({
            'config': cfg_id, 'repr_mode': repr_mode,
            'train_wall_s_mean': np.mean(walls) if walls else float('nan'),
            'train_wall_s_std':  np.std(walls)  if walls else float('nan'),
            'peak_gpu_mb_mean':  np.mean(mems)  if mems  else float('nan'),
            'lat_enroll_s_mean': np.mean(lat_en) if lat_en else float('nan'),
            'lat_probe_s_mean':  np.mean(lat_pr) if lat_pr else float('nan'),
            'disk_footprint_mb': disk_mb,
            'preprocess_load_s': prep_s,
        })
    _tmp = str(_CK) + '.tmp'
    with open(_tmp, 'wb') as _f: pickle.dump(speed_rows, _f)
    os.replace(_tmp, _CK)   # tulis ATOMIK -> cache tak pernah setengah-jadi
    print('[checkpoint SAVE] speed_rows')
df_speed = pd.DataFrame(speed_rows)
df_speed.to_csv(ANA_DIR / 'speed_resource.csv', index=False)
print(df_speed.to_string(index=False))
print(f'\nDisimpan: {ANA_DIR}/speed_resource.csv')

## 12. ROBUSTNESS — Rotation Sensitivity (E10, uji H4/H8)

**Desain yang benar (pipeline-level):** rotasi disuntikkan ke **scan mentah** (raw PLY), lalu tiap representasi menurunkan inputnya seperti di preprocessing:
- **R1 (raw_ply):** pakai titik mentah yang sudah dirotasi → **tidak ada kanonikalisasi** → model lihat pose berbeda.
- **R2/R3 (canonical/fps):** jalankan ulang `pca_align` + unit-sphere pada raw yang dirotasi. Karena PCA align bersifat **rotation-invariant by construction**, pose kanonis pulih → model lihat input ~sama.

Ekspektasi: **ΔEER R1 naik tajam terhadap θ; R2/R3 ~datar.** Inilah bukti kuantitatif paling tajam bahwa kanonikalisasi PCA kritis (H4/H8).

> Catatan: replika `pca_align`/unit-sphere di sini sengaja **identik** dengan `3DRegistration/preprocess_for_cnn.py` agar setara cara file di-generate. Karena R2 & R3 memakai kanonikalisasi yang sama, kurva keduanya akan ~berimpit (keduanya invarian) — itu memang yang diharapkan; ketidakstabilan tanda sumbu-X (~16%, lihat laporan) menambah sedikit noise, bukan tren.


In [ ]:
ROT_ANGLES = [0, 30, 60, 90, 180]   # derajat, sumbu vertikal (z)

# --- Replika pipeline preprocessing (numpy, identik dgn 3DRegistration/preprocess_for_cnn.py) ---
def _pca_align_np(xyz, nrm):
    """PCA align rotation-invariant: center + rotate ke principal axes + flip +Y. Points & normals."""
    centroid = xyz.mean(axis=0)
    centered = xyz - centroid
    _, _, Vt = np.linalg.svd(centered, full_matrices=False)
    r0 = float(np.ptp(centered @ Vt[0])); r1 = float(np.ptp(centered @ Vt[1]))
    if r0 >= r1:
        y_axis, x_axis = Vt[0], Vt[1]
    else:
        y_axis, x_axis = Vt[1], Vt[0]
    z_axis = np.cross(x_axis, y_axis); z_axis /= np.linalg.norm(z_axis)
    x_axis = np.cross(y_axis, z_axis); x_axis /= np.linalg.norm(x_axis)
    R = np.stack([x_axis, y_axis, z_axis], axis=0)
    al  = centered @ R.T
    aln = nrm @ R.T
    flip = np.sum(al[:, 1] > np.median(al[:, 1])) < len(al) // 2
    if flip:
        al[:, 0] *= -1; al[:, 1] *= -1
        aln[:, 0] *= -1; aln[:, 1] *= -1
    return al.astype(np.float32), aln.astype(np.float32)

def _unit_sphere_np(xyz):
    """Scale titik agar masuk unit sphere (normals tidak diubah)."""
    scale = np.max(np.linalg.norm(xyz, axis=1))
    if scale < 1e-8:
        return xyz.astype(np.float32)
    return (xyz / scale).astype(np.float32)

def _rotz(xyz, deg):
    """Rotasi xyz mengelilingi sumbu z (vertikal) sebesar deg derajat."""
    th = np.deg2rad(deg)
    c, s = np.cos(th), np.sin(th)
    R = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    return xyz @ R.T

def _derive_repr(raw6, repr_mode):
    """Dari raw PLY (N,6) yang SUDAH dirotasi → turunkan input model sesuai repr_mode."""
    xyz, nrm = raw6[:, :3].astype(np.float32), raw6[:, 3:6].astype(np.float32)
    if repr_mode == 'raw_ply':
        return raw6.astype(np.float32)                 # tanpa kanonikalisasi
    # canonical_npy & fps_npy: kanonikalisasi ulang (PCA invarian rotasi)
    al, aln = _pca_align_np(xyz, nrm)
    al = _unit_sphere_np(al)
    return np.concatenate([al, aln], axis=1).astype(np.float32)

def _encode_session_rotated(model, session_dirs, repr_mode, normalizer, n_frames, deg, rng):
    """Selalu load RAW ply, rotasi raw, lalu turunkan repr_mode → encode. Fuse mean per sesi."""
    from utils.eval_multiframe import _sample_n_frames, fuse_embeddings
    embs, labs = [], []
    model.eval()
    for label, sdirs in session_dirs.items():
        for sdir in sdirs:
            frame_dirs = _sample_n_frames(sdir, n_frames, seed=42)
            per_frame = []
            for fd in frame_dirs:
                try:
                    raw6, geom = load_session(Path(fd), repr_mode='raw_ply')   # SELALU raw
                    raw6 = raw6[:, :6].astype(np.float32).copy()
                    if deg != 0:
                        raw6[:, :3] = _rotz(raw6[:, :3], deg)
                        raw6[:, 3:6] = _rotz(raw6[:, 3:6], deg)               # rotasi normals juga
                    pts = _derive_repr(raw6, repr_mode)
                    if len(pts) >= N_POINTS:
                        idx = rng.choice(len(pts), N_POINTS, replace=False)
                    else:
                        idx = rng.choice(len(pts), N_POINTS, replace=True)
                    pts = pts[idx]
                    geom = geom.astype(np.float32)
                    if normalizer is not None:
                        geom = normalizer.transform(geom)
                    pts_t  = torch.from_numpy(np.ascontiguousarray(pts)).unsqueeze(0).to(DEVICE)
                    geom_t = torch.from_numpy(geom).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        e = model.encoder(pts_t, geom_t).squeeze(0).cpu().numpy()
                    per_frame.append(e)
                except Exception as ex:
                    print(f'  [WARN] rot encode {fd}: {ex}')
            if per_frame:
                embs.append(fuse_embeddings(np.stack(per_frame), 'mean')); labs.append(label)
    return np.array(embs), labs

def _eer_from_embs(gallery_embs, gallery_labels, probe_embs, probe_labels):
    gen, imp = genuine_impostor_scores(gallery_embs, gallery_labels, probe_embs, probe_labels)
    if len(gen) == 0 or len(imp) == 0:
        return float('nan')
    _, _, _, _, eer = det_roc_curves(gen, imp)
    return eer

import pickle, os
CACHE_DIR = RUNS_DIR / '_eval_cache'; CACHE_DIR.mkdir(parents=True, exist_ok=True)
_CK = CACHE_DIR / 'rot_rows.pkl'

# RESUME per-config: muat partial; skip config yg sudah ada
rot_rows = []
if _CK.exists():
    try:
        _o = pickle.load(open(_CK, 'rb'))
        if isinstance(_o, list): rot_rows = _o
        print(f'[resume] rot_rows: {len({r["config"] for r in rot_rows})} config dimuat')
    except Exception as _e:
        print(f'[corrupt] rot_rows: {_e} -> mulai bersih'); rot_rows = []

def _save_rot():
    _tmp = str(_CK) + '.tmp'
    with open(_tmp, 'wb') as _f: pickle.dump(rot_rows, _f)
    os.replace(_tmp, _CK)   # atomik

_done_cfg = {r['config'] for r in rot_rows}
for cfg_id, repr_mode in REPR_CONFIGS:
    if cfg_id in _done_cfg:
        print(f'  {cfg_id}: sudah ada (resume), skip rotation'); continue
    model, normalizer = load_model(cfg_id, SEEDS[1] if len(SEEDS) > 1 else SEEDS[0])
    if model is None:
        for s in SEEDS:
            model, normalizer = load_model(cfg_id, s)
            if model is not None:
                break
    if model is None:
        print(f'  {cfg_id}: tidak ada checkpoint, skip rotation'); continue
    rng = np.random.default_rng(42)
    g_embs, g_labs = _encode_session_rotated(model, all_session_splits['train'],
                                             repr_mode, normalizer, N_BEST, 0, rng)
    base_eer = None
    for deg in ROT_ANGLES:
        rng = np.random.default_rng(123)
        p_embs, p_labs = _encode_session_rotated(model, all_session_splits['test'],
                                                 repr_mode, normalizer, M_BEST, deg, rng)
        eer = _eer_from_embs(g_embs, g_labs, p_embs, p_labs)
        if deg == 0:
            base_eer = eer
        rot_rows.append({'config': cfg_id, 'repr_mode': repr_mode, 'theta_deg': deg,
                         'eer': eer, 'delta_eer': (eer - base_eer) if base_eer is not None else float('nan')})
        print(f'  {cfg_id} theta={deg:3d}  EER={eer:.4f}')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _save_rot()                                                  # checkpoint disk per config
    git_save(f'v8b [auto] eval §12 rotation {cfg_id}', push=True)   # commit bertahap
df_rot = pd.DataFrame(rot_rows)
df_rot.to_csv(ANA_DIR / 'rotation_sensitivity.csv', index=False)

plt.figure(figsize=(8, 5.5))
for cfg_id, repr_mode in REPR_CONFIGS:
    sub = df_rot[df_rot['config'] == cfg_id]
    if not sub.empty:
        plt.plot(sub['theta_deg'], sub['delta_eer'], marker='o', label=f'{cfg_id} ({repr_mode})')
plt.xlabel('Rotasi raw scan θ (derajat, sumbu vertikal)'); plt.ylabel('Δ EER vs θ=0')
plt.title('Rotation Sensitivity pipeline-level (N=5, M=5)'); plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.savefig(ANA_DIR / 'rotation_sensitivity.png', bbox_inches='tight')
plt.show()
print('Disimpan: rotation_sensitivity.{png,csv}')
print('Ekspektasi: R1 (raw_ply) naik tajam; R2/R3 ~datar (PCA invarian rotasi).')


## 13. ROBUSTNESS — Determinism (E10)

R2 vs R3: encode frame **yang sama** K=10 kali, ukur std embedding lintas pengulangan.
R3 (FPS fixed 8192) → ~0; R2 (random sample) → >0. Simpan `determinism.csv`.

In [ ]:
K_REPEAT = 10

def _collect_some_frames(n=8):
    fds = []
    for lab, sdirs in all_session_splits['test'].items():
        for sdir in sdirs:
            for fd in sorted(Path(sdir).glob('frame_*')):
                if (fd / 'geometry.json').exists():
                    fds.append(fd); break
            if len(fds) >= n: break
        if len(fds) >= n: break
    return fds

import pickle, os
CACHE_DIR = RUNS_DIR / '_eval_cache'; CACHE_DIR.mkdir(parents=True, exist_ok=True)
_CK = CACHE_DIR / 'det_rows.pkl'
_cfg_det = {c for c, r in REPR_CONFIGS if r in ('canonical_npy','fps_npy') and any((RUNS_DIR / c / f'seed_{s}' / 'best.pth').exists() for s in SEEDS)}
det_rows = None
if _CK.exists():
    try:
        _o = pickle.load(open(_CK, 'rb'))
        if isinstance(_o, list) and {r['config'] for r in _o} >= _cfg_det:
            det_rows = _o; print('[checkpoint HIT] det_rows - selesai, skip')
        else:
            print('[checkpoint INVALID] det_rows - tak lengkap, hitung ulang')
    except Exception as _e:
        print(f'[checkpoint CORRUPT] det_rows: {_e} - hitung ulang')
else:
    print('[checkpoint MISS] det_rows - belum ada, mulai dari awal')
if det_rows is None:
    det_rows = []
    det_frames = _collect_some_frames(8)
    for cfg_id, repr_mode in REPR_CONFIGS:
        if repr_mode not in ('canonical_npy', 'fps_npy'):
            continue  # determinism relevan utk R2 vs R3
        model, normalizer = None, None
        for s in SEEDS:
            model, normalizer = load_model(cfg_id, s)
            if model is not None: break
        if model is None:
            print(f'  {cfg_id}: tidak ada checkpoint, skip determinism'); continue
        per_frame_std = []
        model.eval()
        for fd in det_frames:
            reps = []
            for k in range(K_REPEAT):
                try:
                    pts, geom = load_session(Path(fd), repr_mode=repr_mode)
                    pts = pts[:, :6].astype(np.float32)
                    rng = np.random.default_rng(1000 + k)  # seed berbeda tiap repeat
                    if len(pts) >= N_POINTS:
                        idx = rng.choice(len(pts), N_POINTS, replace=False)
                    else:
                        idx = rng.choice(len(pts), N_POINTS, replace=True)
                    pts = pts[idx]
                    geom = geom.astype(np.float32)
                    if normalizer is not None:
                        geom = normalizer.transform(geom)
                    pts_t  = torch.from_numpy(pts).unsqueeze(0).to(DEVICE)
                    geom_t = torch.from_numpy(geom).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        e = model.encoder(pts_t, geom_t).squeeze(0).cpu().numpy()
                    reps.append(e)
                except Exception as ex:
                    print(f'  [WARN] det encode {fd}: {ex}')
            if len(reps) >= 2:
                reps = np.stack(reps)                 # (K, D)
                per_frame_std.append(reps.std(axis=0).mean())
        mean_std = float(np.mean(per_frame_std)) if per_frame_std else float('nan')
        det_rows.append({'config': cfg_id, 'repr_mode': repr_mode,
                         'mean_embedding_std': mean_std, 'n_frames': len(per_frame_std)})
        print(f'  {cfg_id} ({repr_mode}): mean embedding std lintas {K_REPEAT} repeat = {mean_std:.6f}')
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    _tmp = str(_CK) + '.tmp'
    with open(_tmp, 'wb') as _f: pickle.dump(det_rows, _f)
    os.replace(_tmp, _CK)   # tulis ATOMIK -> cache tak pernah setengah-jadi
    print('[checkpoint SAVE] det_rows')
df_det = pd.DataFrame(det_rows)
df_det.to_csv(ANA_DIR / 'determinism.csv', index=False)
print(df_det.to_string(index=False))
print(f'\nDisimpan: {ANA_DIR}/determinism.csv')

## 14. SYNTHESIS — Pareto (E11)

Scatter Pareto: x=latency probe (atau train wall), y=EER (N5M5), satu titik per config.
Tandai frontier Pareto. Tabel gabungan accuracy×speed×disk. Simpan `pareto.png` + `summary_v8b_arclab.csv`.

In [ ]:
# EER N5M5 mean per config
def eer_n5m5(cfg_id):
    eers = []
    for seed in SEEDS:
        e = ablation_results.get((cfg_id, seed), {}).get((N_BEST, M_BEST), {}).get('eer', float('nan'))
        if not np.isnan(e): eers.append(e)
    return (np.mean(eers), np.std(eers)) if eers else (float('nan'), float('nan'))

summary_rows = []
for cfg_id, repr_mode in REPR_CONFIGS:
    eer_m, eer_s = eer_n5m5(cfg_id)
    srow = df_speed[df_speed['config'] == cfg_id]
    s = srow.iloc[0] if not srow.empty else {}
    summary_rows.append({
        'config': cfg_id, 'repr_mode': repr_mode,
        'eer_n5m5_mean': eer_m, 'eer_n5m5_std': eer_s,
        'auc_n5m5': auc_summary.get(cfg_id, {}).get('auc', float('nan')),
        'lat_probe_s': float(s.get('lat_probe_s_mean', float('nan'))) if len(s) else float('nan'),
        'train_wall_s': float(s.get('train_wall_s_mean', float('nan'))) if len(s) else float('nan'),
        'disk_footprint_mb': float(s.get('disk_footprint_mb', float('nan'))) if len(s) else float('nan'),
        'preprocess_load_s': float(s.get('preprocess_load_s', float('nan'))) if len(s) else float('nan'),
    })
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(ANA_DIR / 'summary_v8b_arclab.csv', index=False)
print(df_summary.to_string(index=False))

# Pareto frontier (minimize x=lat_probe, y=eer)
def pareto_mask(xs, ys):
    n = len(xs); mask = [True] * n
    for i in range(n):
        for j in range(n):
            if j == i: continue
            if (xs[j] <= xs[i] and ys[j] <= ys[i]) and (xs[j] < xs[i] or ys[j] < ys[i]):
                mask[i] = False; break
    return mask

valid = df_summary.dropna(subset=['lat_probe_s', 'eer_n5m5_mean'])
plt.figure(figsize=(8, 6))
if not valid.empty:
    xs = valid['lat_probe_s'].to_numpy(); ys = valid['eer_n5m5_mean'].to_numpy()
    labels = valid['config'].to_list(); reprs = valid['repr_mode'].to_list()
    mask = pareto_mask(xs.tolist(), ys.tolist())
    plt.scatter(xs, ys, s=120, c=['tab:red' if m else 'tab:blue' for m in mask], zorder=3)
    for x, y, lab, rp in zip(xs, ys, labels, reprs):
        plt.annotate(f'{lab}\n{rp}', (x, y), textcoords='offset points', xytext=(8, 6), fontsize=9)
    # garis frontier
    fr = sorted([(x, y) for x, y, m in zip(xs, ys, mask) if m])
    if len(fr) >= 2:
        plt.plot([p[0] for p in fr], [p[1] for p in fr], 'r--', alpha=0.6, zorder=2)
plt.xlabel('Latency probe (s, N=5)'); plt.ylabel('EER (N=5, M=5)')
plt.title('Pareto: Accuracy vs Speed per Representasi'); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ANA_DIR / 'pareto.png', bbox_inches='tight')
plt.show()
print(f'Disimpan: {ANA_DIR}/pareto.png + summary_v8b_arclab.csv')

## 15. SUMMARY.md (Bahasa Indonesia)

In [ ]:
lines = [
    '# v8b Representation Ablation — Summary',
    '',
    f'**Tanggal**: {TS}',
    f'**Setup**: 3 representasi (R1/R2/R3), loss DIKUNCI arcface_m04 (m={ARC_MARGIN}, s={ARC_SCALE}), '
    f'closed-set (Test+Holdout, tanpa LOSO).',
    f'**Seeds**: {SEEDS}  |  N_POINTS={N_POINTS}  |  frame_mode={FRAME_MODE}',
    f'**Protokol akurasi primer**: multi-frame fusion N={N_BEST}, M={M_BEST}, strategy=mean.',
    '',
    '## 1. Akurasi per Config (EER N=5, M=5)',
    '',
    '| Config | Repr | EER (mean±std) | AUC |',
    '|--------|------|----------------|-----|',
]
for _, r in df_summary.iterrows():
    eer = f"{r['eer_n5m5_mean']:.4f}±{r['eer_n5m5_std']:.4f}" if not np.isnan(r['eer_n5m5_mean']) else 'N/A'
    auc = f"{r['auc_n5m5']:.4f}" if not np.isnan(r['auc_n5m5']) else 'N/A'
    lines.append(f"| {r['config']} | {r['repr_mode']} | {eer} | {auc} |")

lines += ['', '## 2. Speed & Resource', '',
          '| Config | Repr | Train wall (s) | Peak GPU (MB) | Lat probe (s) | Disk (MB) | Preproc load (s) |',
          '|--------|------|----------------|---------------|---------------|-----------|------------------|']
for _, r in df_speed.iterrows():
    def _f(x, p=2):
        return f'{x:.{p}f}' if (x is not None and not (isinstance(x, float) and np.isnan(x))) else 'N/A'
    lines.append(f"| {r['config']} | {r['repr_mode']} | {_f(r['train_wall_s_mean'],1)} | "
                 f"{_f(r['peak_gpu_mb_mean'],1)} | {_f(r['lat_probe_s_mean'],3)} | "
                 f"{_f(r['disk_footprint_mb'],1)} | {_f(r['preprocess_load_s'],4)} |")

# Verdict rotasi
lines += ['', '## 3. Verdict Robustness — Rotasi']
if not df_rot.empty:
    lines.append('')
    lines.append('| Config | Repr | ΔEER @θ=180° |')
    lines.append('|--------|------|-------------|')
    for cfg_id, repr_mode in REPR_CONFIGS:
        sub = df_rot[(df_rot['config'] == cfg_id) & (df_rot['theta_deg'] == 180)]
        d = sub['delta_eer'].iloc[0] if not sub.empty else float('nan')
        lines.append(f'| {cfg_id} | {repr_mode} | {d:+.4f} |' if not np.isnan(d) else f'| {cfg_id} | {repr_mode} | N/A |')
    lines.append('')
    lines.append('_Ekspektasi: R1 (raw_ply) ΔEER besar (sensitif rotasi); R2/R3 ~datar (PCA rotation-invariant)._')

# Determinism
lines += ['', '## 4. Determinism (R2 vs R3)']
if not df_det.empty:
    lines += ['', '| Config | Repr | Mean embedding std (K=10) |', '|--------|------|---------------------------|']
    for _, r in df_det.iterrows():
        lines.append(f"| {r['config']} | {r['repr_mode']} | {r['mean_embedding_std']:.6f} |")
    lines.append('')
    lines.append('_R3 (FPS fixed) seharusnya ~0; R2 (random sample) > 0._')

# Pareto winner
lines += ['', '## 5. Pareto Winner']
valid2 = df_summary.dropna(subset=['eer_n5m5_mean'])
if not valid2.empty:
    win = valid2.loc[valid2['eer_n5m5_mean'].idxmin()]
    lines.append('')
    lines.append(f"Config dengan EER terendah: **{win['config']} ({win['repr_mode']})** "
                 f"EER={win['eer_n5m5_mean']:.4f}, lat_probe={win['lat_probe_s']:.3f}s, "
                 f"disk={win['disk_footprint_mb']:.1f}MB.")
    lines.append('')
    lines.append('_Pilih representasi di frontier Pareto (akurasi vs kecepatan vs disk) sesuai prioritas deployment._')

lines += ['', '_Dihasilkan otomatis oleh v8b_arclab_repr_ablation.ipynb_']

(ANA_DIR / 'SUMMARY.md').write_text('\n'.join(lines))
print(f'Summary disimpan: {ANA_DIR}/SUMMARY.md')
print('\n'.join(lines))

## 11b. Track C — EER Terstratifikasi Kualitas (bukti QA-ArcFace)
Bandingkan EER pada subset scan **kualitas-rendah vs tinggi** (median split point_count).
Hipotesis H-C2: QA-ArcFace paling menolong di subset kualitas-rendah.

In [ ]:
# Eval terstratifikasi kualitas (N5M5) per loss. Defensif: skip bila gagal.
import json as _json
def _session_quality(sdir):
    qs = []
    for fd in sorted(Path(sdir).glob('frame_*')):
        g = fd/'geometry.json'
        if g.exists():
            try: qs.append(int(_json.load(open(g)).get('point_count',0)))
            except Exception: pass
    return float(np.mean(qs)) if qs else 0.0

try:
    test_split = all_session_splits['test']
    allq = [( _session_quality(s), lab, s) for lab,ss in test_split.items() for s in ss]
    med = float(np.median([q for q,_,_ in allq])) if allq else 0.0
    low, high = {}, {}
    for q,lab,s in allq:
        (low if q <= med else high).setdefault(lab, []).append(s)
    strat_rows = []
    for cfg_id, _ in REPR_CONFIGS:
        model, normalizer = load_model(cfg_id, SEEDS[0])
        if model is None: continue
        for name, probe in (('low', low), ('high', high)):
            if not probe: continue
            res = eval_multiframe_ablation(model=model, enroll_session_splits=all_session_splits['train'],
                probe_session_splits=probe, n_list=[N_BEST], m_list=[M_BEST], fusion_strategy='mean',
                device=DEVICE, n_points=N_POINTS, normalizer=normalizer, seed=SEEDS[0], repr_mode=FIXED_ALIGN)
            eer = res.get((N_BEST,M_BEST),{}).get('eer', float('nan'))
            strat_rows.append({'loss':cfg_id,'stratum':name,'eer':eer})
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    df_strat = pd.DataFrame(strat_rows)
    df_strat.to_csv(ANA_DIR/'eer_by_quality.csv', index=False)
    print(df_strat.to_string(index=False))
    print(f'median point_count split = {med:.0f}')
except Exception as e:
    print('[stratified eval skip]', e)


## 16. Final Git Save

In [ ]:
git_save('v8b: repr ablation + analysis complete', push=True)
print('\nSelesai! Lihat', ANA_DIR)

## 17. (Opsional) Arsip runs → GitHub Release `runs-v8`
Setelah ablation selesai: kemas checkpoint (dari Drive) jadi satu artifak Release. Repo tetap ramping; model tetap tersimpan permanen.

In [ ]:
# Arsip checkpoint (RUNS_DIR di Drive) → aset Release (split >1.9GB otomatis).
# git tidak menyimpan .pth; ini menjadikannya artifak permanen di luar repo.
import sys as _s, subprocess as _sp, glob as _g, shutil as _sh
_s.path.insert(0, str(REPO_DIR / '3DRegistration'))
import importlib, release_assets as _ra; importlib.reload(_ra)

RUNS_TAG = 'runs-v8'
_arc = Path('/content/_runs_archive'); _sh.rmtree(_arc, ignore_errors=True); _arc.mkdir()
_tar = _arc / 'runs_v8.tar.zst'
print('kemas', RUNS_DIR, '...')
_sp.run(['tar', '--use-compress-program', 'zstd -T0', '-cf', str(_tar),
         '-C', str(RUNS_DIR.parent), RUNS_DIR.name], check=True)
_LIM = 1900*1024*1024
if _tar.stat().st_size > _LIM:
    _sp.run(['split','-b',str(_LIM),'-d','-a','2',str(_tar),str(_tar)+'.part'], check=True)
    _tar.unlink(); _files = sorted(str(p) for p in _arc.glob('runs_v8.tar.zst.part*'))
else:
    _files = [str(_tar)]
_ra.create_or_get_release(REPO_SLUG, RUNS_TAG, GITHUB_TOKEN,
                          name='v8 training runs (checkpoints)',
                          body='Checkpoint hasil ablation v8 (dari Drive). Repo git tetap ramping.',
                          prerelease=True)
for _f in _files:
    _ra.upload_asset(REPO_SLUG, RUNS_TAG, _f, GITHUB_TOKEN)
print('✅ runs ter-arsip ke Release', RUNS_TAG)
